# Run a calibration

Run `qdash config init --project-id your-project-id` in a terminal, then fill in `api_token` in the generated file. Use `--base-url` for a remote deployment. If the file already exists, edit it instead. `QDash()` reads its `default_profile`; use `QDash.from_profile("local")` to select a profile explicitly. See the [Python reference](../../../docs/reference/clients/python.md#connect-to-a-deployment) for setup. Replace the chip ID and choose settings suitable for your hardware.

In [ ]:
from qdash.sdk import QDash
from qdash.sdk.calibration import CustomOneQubit, Pipeline, QubitTargets, TaskCall, sweep

qdash = QDash()
chip = qdash.chip("YOUR_CHIP_ID")

## Define Calibration

In [ ]:
pipeline = Pipeline(
    name="coherence",
    targets=QubitTargets(["0", "1"]),
    steps=[
        CustomOneQubit(
            step_name="t1",
            tasks=[
                TaskCall(name="Configure"),
                TaskCall(name="CheckT1", run={"time_range": sweep.linspace(300, 500_000, 51)}),
            ],
        ),
    ],
    default_run_parameters={"readout_duration": 1024, "interval": 262_144},
)

## Validate Calibration

Validation does not start a measurement. Review any problems before continuing.

In [ ]:
check = chip.validate_calibration(pipeline)
print("Valid:", check.valid)
for problem in check.problems or []:
    print(problem.path, problem.message)
for warning in check.warnings or []:
    print("Warning:", warning.path, warning.message)

## Run Calibration

Running this cell starts a measurement; re-running it starts another. A wait timeout does not cancel it. Resume tracking with `qdash.run(run.execution_id)`.

In [ ]:
run = chip.calibrate(pipeline)
print("Execution:", run.execution_id)
result = run.wait(timeout_seconds=600)
print(result.status)

## Close Connection

In [ ]:
qdash.close()